# Speech denoising with TIMIT, Gaussian noise, and a SimpleRNN

This notebook uses **PyTorch** to train a simple recurrent neural network (`torch.nn.RNN`) that estimates clean speech from speech corrupted with Gaussian noise.

Pipeline: `clean audio → Gaussian noise → STFT → SimpleRNN → estimated STFT → denoised audio`.

The existing `Train`, `Val`, and `Test` folders are kept separate throughout the experiment.

## 2. Imports and configuration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, display
from torch import nn
from torch.utils.data import DataLoader, Dataset

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# TIMIT audio settings
SAMPLE_RATE = 16_000
# The RNN sees five seconds at a time during both training and inference.
SEGMENT_SECONDS = 5.0
SEGMENT_SAMPLES = int(SAMPLE_RATE * SEGMENT_SECONDS)

# STFT settings
FRAME_LENGTH = 256
FRAME_STEP = 128
FFT_LENGTH = 256
N_FREQUENCIES = FFT_LENGTH // 2 + 1

# Training settings
RNN_UNITS = 128
# Five-second sequences are longer, so a moderate batch size is safer.
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 1e-3
MIN_SNR_DB = 0.0
MAX_SNR_DB = 15.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Expected project structure:
# RNN/
# ├── simple_rnn_timit_denoising.ipynb
# └── Timit/
#     ├── Train/
#     ├── Val/
#     └── Test/
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "Timit"
TRAIN_DIR = DATA_DIR / "Train"
VAL_DIR = DATA_DIR / "Val"
TEST_DIR = DATA_DIR / "Test"
OUTPUT_DIR = PROJECT_DIR / "outputs"

print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"Dataset folder: {DATA_DIR.resolve()}")

## 3. Load the existing dataset splits

Each folder is read independently. The notebook never creates a new random split and never uses test data during training.

In [ ]:
def find_wav_files(folder):
    """Find all WAV files inside one split folder."""
    if not folder.exists():
        raise FileNotFoundError(f"Split folder not found: {folder.resolve()}")

    files = sorted(
        path for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() == ".wav"
    )

    if not files:
        raise FileNotFoundError(f"No WAV files found in: {folder.resolve()}")

    return files


train_files = find_wav_files(TRAIN_DIR)
validation_files = find_wav_files(VAL_DIR)
test_files = find_wav_files(TEST_DIR)

# File names identify speakers in this version of the dataset.
train_ids = {path.stem.lower() for path in train_files}
validation_ids = {path.stem.lower() for path in validation_files}
test_ids = {path.stem.lower() for path in test_files}

assert train_ids.isdisjoint(validation_ids), "Train/Val speaker overlap detected."
assert train_ids.isdisjoint(test_ids), "Train/Test speaker overlap detected."
assert validation_ids.isdisjoint(test_ids), "Val/Test speaker overlap detected."

print(f"Train files:      {len(train_files)}")
print(f"Validation files: {len(validation_files)}")
print(f"Test files:       {len(test_files)}")
print("No speaker/file-ID overlap was found between splits.")

## 4. Audio and Gaussian-noise functions

Every recording is divided into five-second segments. Segments are loaded only when a batch needs them, which avoids keeping several gigabytes of waveforms and spectrograms in memory.

In [ ]:
def load_audio_segment(path, start_frame):
    """Load and normalize one fixed-length mono segment."""
    audio, sample_rate = sf.read(
        path,
        start=start_frame,
        frames=SEGMENT_SAMPLES,
        dtype="float32",
        always_2d=False,
    )

    if sample_rate != SAMPLE_RATE:
        raise ValueError(
            f"{path} has a sample rate of {sample_rate} Hz; "
            f"expected {SAMPLE_RATE} Hz."
        )

    # Convert stereo to mono if necessary.
    if audio.ndim == 2:
        audio = audio.mean(axis=1)

    clean_audio = torch.from_numpy(audio)

    # Pad the final segment with silence.
    if len(clean_audio) < SEGMENT_SAMPLES:
        clean_audio = nn.functional.pad(
            clean_audio,
            (0, SEGMENT_SAMPLES - len(clean_audio)),
        )

    # Peak normalization, protected against silent segments.
    peak = clean_audio.abs().max().clamp_min(1e-8)
    return clean_audio / peak


def add_gaussian_noise(clean_audio, snr_db, generator=None):
    """Add Gaussian noise at a specified signal-to-noise ratio."""
    signal_power = clean_audio.square().mean()
    noise_power = signal_power / (10.0 ** (snr_db / 10.0))

    noise = torch.randn(
        clean_audio.shape,
        dtype=clean_audio.dtype,
        generator=generator,
    )
    noise = noise * torch.sqrt(noise_power + 1e-10)

    return (clean_audio + noise).clamp(-1.0, 1.0)


def random_snr(generator=None):
    """Sample one SNR value between the configured limits."""
    return torch.empty(1).uniform_(
        MIN_SNR_DB,
        MAX_SNR_DB,
        generator=generator,
    ).item()

## 5. STFT and PyTorch Dataset

The STFT converts each waveform into `[time, frequency]` features. 

In [ ]:
STFT_WINDOW = torch.hann_window(FRAME_LENGTH)


def stft(audio):
    """Compute the complex STFT of an audio tensor."""
    return torch.stft(
        audio,
        n_fft=FFT_LENGTH,
        hop_length=FRAME_STEP,
        win_length=FRAME_LENGTH,
        window=STFT_WINDOW.to(audio.device),
        return_complex=True,
    )


def magnitude_spectrogram(audio):
    """Return the STFT magnitude in [time, frequency] order."""
    return stft(audio).abs().transpose(0, 1)


class TIMITDenoisingDataset(Dataset):
    """Create noisy/clean spectrogram pairs from a list of WAV files."""

    def __init__(self, wav_files, deterministic_noise=False):
        self.segments = []
        self.deterministic_noise = deterministic_noise

        # Store lightweight (file, start-frame) references, not full audio.
        for path in wav_files:
            info = sf.info(path)
            if info.samplerate != SAMPLE_RATE:
                raise ValueError(
                    f"{path} has {info.samplerate} Hz; expected {SAMPLE_RATE} Hz."
                )

            for start in range(0, info.frames, SEGMENT_SAMPLES):
                self.segments.append((path, start))

    def __len__(self):
        return len(self.segments)

    def get_audio_pair(self, index):
        """Return noisy and clean waveforms for one segment."""
        path, start = self.segments[index]
        clean_audio = load_audio_segment(path, start)

        generator = None
        if self.deterministic_noise:
            generator = torch.Generator().manual_seed(SEED + index)

        snr_db = random_snr(generator)
        noisy_audio = add_gaussian_noise(clean_audio, snr_db, generator)
        return noisy_audio, clean_audio

    def __getitem__(self, index):
        noisy_audio, clean_audio = self.get_audio_pair(index)
        noisy_magnitude = magnitude_spectrogram(noisy_audio)
        clean_magnitude = magnitude_spectrogram(clean_audio)
        return noisy_magnitude, clean_magnitude


train_dataset = TIMITDenoisingDataset(
    train_files,
    deterministic_noise=False,
)
validation_dataset = TIMITDenoisingDataset(
    validation_files,
    deterministic_noise=True,
)
test_dataset = TIMITDenoisingDataset(
    test_files,
    deterministic_noise=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print(f"Training segments:   {len(train_dataset)}")
print(f"Validation segments: {len(validation_dataset)}")
print(f"Test segments:       {len(test_dataset)}")

## 6. Create the SimpleRNN

`batch_first=True` gives the model `[batch, time, frequency]` inputs. The output layer estimates one clean magnitude value for every time-frequency bin.

In [ ]:
class SimpleRNNDenoiser(nn.Module):
    """A basic recurrent network for spectrogram denoising."""

    def __init__(self, n_frequencies, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=n_frequencies,
            hidden_size=hidden_size,
            batch_first=True,
        )
        self.output_layer = nn.Linear(hidden_size, n_frequencies)

    def forward(self, noisy_magnitude):
        sequence, _ = self.rnn(noisy_magnitude)
        # Spectrogram magnitudes cannot be negative.
        return torch.relu(self.output_layer(sequence))


model = SimpleRNNDenoiser(N_FREQUENCIES, RNN_UNITS).to(DEVICE)

print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 7. Train with Train and select with Validation

The test loader is not accessed during training. Validation loss is used only to monitor generalization and keep the best model weights.

In [ ]:
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


def evaluate(model, data_loader):
    """Compute average MSE without updating model parameters."""
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for noisy_magnitude, clean_magnitude in data_loader:
            noisy_magnitude = noisy_magnitude.to(DEVICE)
            clean_magnitude = clean_magnitude.to(DEVICE)

            prediction = model(noisy_magnitude)
            loss = loss_function(prediction, clean_magnitude)
            total_loss += loss.item() * len(noisy_magnitude)

    return total_loss / len(data_loader.dataset)


train_losses = []
validation_losses = []
best_validation_loss = float("inf")
best_state = None

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0

    for noisy_magnitude, clean_magnitude in train_loader:
        noisy_magnitude = noisy_magnitude.to(DEVICE)
        clean_magnitude = clean_magnitude.to(DEVICE)

        optimizer.zero_grad()
        prediction = model(noisy_magnitude)
        loss = loss_function(prediction, clean_magnitude)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item() * len(noisy_magnitude)

    train_loss = total_train_loss / len(train_dataset)
    validation_loss = evaluate(model, validation_loader)

    train_losses.append(train_loss)
    validation_losses.append(validation_loss)

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_state = {
            name: parameter.detach().cpu().clone()
            for name, parameter in model.state_dict().items()
        }

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} - "
        f"train loss: {train_loss:.6f} - "
        f"validation loss: {validation_loss:.6f}"
    )

# Restore the epoch with the best validation loss.
model.load_state_dict(best_state)

plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="train")
plt.plot(validation_losses, label="validation")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Model loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8. Final evaluation on Test

The test set is evaluated once, after model selection is complete.

In [ ]:
test_loss = evaluate(model, test_loader)
print(f"Final test MSE: {test_loss:.6f}")

## 9. Reconstruct and listen to one five-second window

One five-second window is taken from the test set. Its predicted magnitude is combined with the noisy phase, and `torch.istft` reconstructs the five-second waveform.

In [ ]:
def denoise_window(model, noisy_audio):
    """Estimate and reconstruct one five-second audio window."""
    model.eval()
    noisy_audio = noisy_audio.to(DEVICE)
    noisy_stft = stft(noisy_audio)
    noisy_magnitude = noisy_stft.abs()

    # RNN input: [batch, time, frequency]
    rnn_input = noisy_magnitude.transpose(0, 1).unsqueeze(0)

    with torch.no_grad():
        predicted_magnitude = model(rnn_input)[0].transpose(0, 1)

    # Reuse the normalized complex phase of the noisy signal.
    noisy_phase = noisy_stft / (noisy_magnitude + 1e-8)
    predicted_stft = predicted_magnitude * noisy_phase

    reconstructed = torch.istft(
        predicted_stft,
        n_fft=FFT_LENGTH,
        hop_length=FRAME_STEP,
        win_length=FRAME_LENGTH,
        window=STFT_WINDOW.to(DEVICE),
        length=SEGMENT_SAMPLES,
    )
    return reconstructed.clamp(-1.0, 1.0).cpu()


# Use one deterministic five-second window from Test, never from Train.
example_path, example_start = test_dataset.segments[0]
noisy_example, clean_example = test_dataset.get_audio_pair(0)
denoised_example = denoise_window(model, noisy_example)

start_seconds = example_start / SAMPLE_RATE
end_seconds = start_seconds + SEGMENT_SECONDS
print(f"Test file: {example_path.name}")
print(f"Window: {start_seconds:.1f} to {end_seconds:.1f} seconds")

print("Clean five-second test window")
display(Audio(clean_example.numpy(), rate=SAMPLE_RATE))

print("Five-second test window with Gaussian noise")
display(Audio(noisy_example.numpy(), rate=SAMPLE_RATE))

print("Five-second test window estimated by the SimpleRNN")
display(Audio(denoised_example.numpy(), rate=SAMPLE_RATE))

## 10. Plot and save the result

In [ ]:
time = np.arange(SEGMENT_SAMPLES) / SAMPLE_RATE

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
signals = [
    (clean_example.numpy(), "Clean five-second test window"),
    (noisy_example.numpy(), "Five-second window with Gaussian noise"),
    (denoised_example.numpy(), "Five-second window estimated by SimpleRNN"),
]

for axis, (signal, title) in zip(axes, signals):
    axis.plot(time, signal, linewidth=0.7)
    axis.set_title(title)
    axis.set_ylabel("Amplitude")
    axis.grid(alpha=0.2)

axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

OUTPUT_DIR.mkdir(exist_ok=True)
audio_path = OUTPUT_DIR / f"denoised_{example_path.stem}_5_seconds.wav"
sf.write(audio_path, denoised_example.numpy(), SAMPLE_RATE)

print(f"Audio saved to: {audio_path.resolve()}")

## 11. Save the PyTorch model

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
model_path = OUTPUT_DIR / "simple_rnn_denoiser.pt"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "n_frequencies": N_FREQUENCIES,
        "hidden_size": RNN_UNITS,
        "sample_rate": SAMPLE_RATE,
        "segment_seconds": SEGMENT_SECONDS,
        "frame_length": FRAME_LENGTH,
        "frame_step": FRAME_STEP,
        "fft_length": FFT_LENGTH,
        "min_snr_db": MIN_SNR_DB,
        "max_snr_db": MAX_SNR_DB,
        "test_mse": test_loss,
        "train_losses": train_losses,
        "validation_losses": validation_losses,
    },
    model_path,
)

print(f"Model saved to: {model_path.resolve()}")